<center>
    
# ORQUESTACION - PIPELINE ANALISIS DE COBRANZAS
</center>

## <u> 1. Configuración de entorno
    Se instalan e importan las dependencias requeridas:
        - Librerías y módulos estandar
        - Módulos propios que son parte de este desarrollo y se encuentran alojados en el repo GitHub

In [ ]:
# Recupera e instala dependencias desde el archivo "Requirements.txt" desde el repo.

url_requirements = "https://raw.githubusercontent.com/Marcelo-F-Martin/pipeline_analisis_de_cobranzas/refs/heads/main/requirements.txt"
%pip install -q -r {url_requirements}

In [ ]:
import os
import pandas as pd
import requests
import io
import unicodedata #para manejo de tildes
from sqlalchemy import create_engine, text
import pymysql
from pymysql.constants import CLIENT
from dotenv import load_dotenv

In [ ]:
# Recupera los módulos propios desde el repo.

dic_modulos = {
    "extraccion.py": "https://raw.githubusercontent.com/Marcelo-F-Martin/pipeline_analisis_de_cobranzas/refs/heads/main/modulos/extraccion.py",
    "transformacion.py": "https://raw.githubusercontent.com/Marcelo-F-Martin/pipeline_analisis_de_cobranzas/refs/heads/main/modulos/transformacion.py",
    "carga.py": "https://raw.githubusercontent.com/Marcelo-F-Martin/pipeline_analisis_de_cobranzas/refs/heads/main/modulos/carga.py"
}

for nombre, url in dic_modulos.items():
    respuesta = requests.get(url)
    
    if respuesta.status_code == 200:
        with open(nombre, "w", encoding="utf-8") as archivo:
            archivo.write(respuesta.text)

In [ ]:
import extraccion
import transformacion
import carga

## <u>2. Inicio Proceso de ETL
    
#### Flujo de trabajo:
    - Conecta al Repo GitHub, extrae archivos en crudo .xls y genera un único DF
    - Selecciona columnas, normaliza encabezados 
    - Conecta con MySQL e ingesta datos a la BD ejecutando los scripts SQL alojados en el repo GitHub
    - Inicializa el Template de Power Bi Desktop para conectar con BD


##### 2.1. Conecta al Repo GitHub, extrae archivos en crudo .xls y genera un único DF

In [ ]:
# Asigna como valor de la variable una lista de diccionarios que contiene los archivos crudos .xls

archivos_crudos = extraccion.extrae_archivos_del_repo()

In [ ]:
# Esta función devuelve como resultado del proceso de extracción, un único DF

df_final = extraccion.genera_unico_DF(archivos_crudos)

##### 2.2. Selecciona columnas, normaliza encabezados
##### <i><u>Aclaración</u></i>:

    Para garantizar un correcto modelo estrella en la BD, se seleccionan de la tabla fuente, únicamente las columnas que no impliquen redundancia para la tabla final y que contribuyan a un diseño normalizado y eficiente del modelo (Tabla de hechos + Tabla de dimensiones).

In [ ]:
# Aplica funciones de selección de columnas y normalización de encabezados.

df_final = transformacion.seleccionar_columnas(transformacion.normalizar_encabezados(df_final))
df_final.info()

In [ ]:
df_final_limpio = df_final.copy()

##### <i><u>Celda de ejecución opcional</u></i>
    Previo a la ingesta del DF en la BD, se da la opción al usuario de descargar localmente el DF en formato .csv

In [ ]:
# Descomente la siguiente linea en caso de querer descargar el archivo

#carga.guardar_archivo(df_final_limpio)

##### 2.3. Conecta con MySQL e ingesta datos a la BD ejecutando los scripts SQL alojados en el repo GitHub

In [ ]:
# Check de variables de entorno y conexión a MySQL

carga.check_env_y_conn_MySQL()

In [ ]:
# Ejecución de scripts SQL en MySQL

carga.ejecutar_script_sql(df_final_limpio)

##### 2.4. Inicializa el Template de Power Bi Desktop para conectar con BD

In [ ]:
# Inicializa aplicación Power BI.

carga.ini_pbi()